# Errors in code prompts

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import pandas as pd
import os

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.utils import pandas_to_latex
from gsm_benchmarker.benchmark.answer_extractor import AnswerExtractor

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_code_short = pp / 'code_short_full__27_04/final'
p_code_long = pp / 'code_no_sep_full__12_03/corrected'

In [ ]:
mres_code_short = MultiVariantMultiModelResultsAnalyser(p_code_short)
mres_code_long = MultiVariantMultiModelResultsAnalyser(p_code_long)

In [ ]:
mres_code_short.variants['main'].models

In [ ]:
mres_code_long.variants['main'].models

In [ ]:
MODELS = [
    'Mathstral-7B-v0.1',
    'Phi-3-mini-128k-instruct',
    'Phi-3.5-mini-instruct',
    'gemma-2-9b'
]

In [ ]:
ERROR_COLOURS = {
    'NAME_ERROR': 'tab:blue',
    'TYPE_VALUE_ERROR': 'tab:green',
    'SYNTAX_ERROR': 'tab:red',
    'NONE_RETURNED': 'tab:orange',
    'NOT_A_NUMBER': 'tab:purple',
    'Other': '#ca9161'              # Brown (for any catch-all category)
}

colours_list = list(ERROR_COLOURS.values())
colours_code_labels = list(ERROR_COLOURS.keys())

ERROR_NAMES = {
    'NAME_ERROR': 'Name E.',
    'TYPE_VALUE_ERROR': 'Type/Value E.',
    'SYNTAX_ERROR': 'Syntax E.',
    'NONE_RETURNED': 'None returned',
    'NOT_A_NUMBER': 'Not a number',
    'Other': '(Other)'
}

In [ ]:
OUTPUTS = Path("./outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
df_short = mres_code_short.variants['main'].get_failed_answer_cases(models=MODELS)
df_long = mres_code_long.variants['main'].get_failed_answer_cases(models=MODELS)

df_short['prompt'] = len(df_short) * ['short']
df_long['prompt'] = len(df_long) * ['long']

df = pd.concat([df_short, df_long], ignore_index=True)
df = df[['model', 'id', 'instance', 'question', 'detected_result_pattern', 'full_response', 'prompt']]
df = df.rename(columns={'detected_result_pattern': 'error_type'})

df

In [ ]:
def add_error_legend(fig_, errors_plotted_):
    errors_plotted_ = set(errors_plotted_)
    legend_patches = [mpatches.Patch(color=color, label=ERROR_NAMES[error_name])
                      for error_name, color in ERROR_COLOURS.items() if error_name in errors_plotted_]
    fig_.legend(handles=legend_patches, loc='lower center', ncol=len(legend_patches), title="Error Type", frameon=True, framealpha=0.5, fontsize=9)


In [ ]:
grouped = df.groupby(['prompt', 'model', 'error_type']).size().unstack(fill_value=0)
grouped = grouped.reindex(columns=ERROR_COLOURS.keys(), fill_value=0)

grouped_pct = grouped.div(grouped.sum(axis=1), axis=0) * 100

prompts = df.prompt.unique().tolist()

errors_plotted = []


fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey='row', sharex='col')
bar_kwargs = dict(edgecolor='white', linewidth=0.5, kind='bar', stacked=True, legend=False)

for i, prompt_type in enumerate(prompts):

    # Extract data for the specific prompt
    count_data = grouped.loc[prompt_type]
    pct_data = grouped_pct.loc[prompt_type]


    # Plot stacked bar charts
    count_data.plot(ax=axes[0, i], color=colours_list, **bar_kwargs)
    pct_data.plot(ax=axes[1, i], color=colours_list, **bar_kwargs)

    # Formatting the subplot
    axes[0, i].set_title(f"{prompt_type.capitalize()} code prompt")
    axes[1, i].set_xlabel("Model")
    axes[1, i].yaxis.set_major_formatter(mtick.PercentFormatter()) # Format Y-axis as percentage
    axes[1, i].tick_params(axis='x', rotation=15)

    totals_per_error = count_data.sum(axis=0)
    errors_plotted.extend(totals_per_error[totals_per_error > 0].index)

    totals_per_model = count_data.sum(axis=1)

    for x_position, total in enumerate(totals_per_model):
        axes[0, i].text(x=x_position, y=total, s=f"{int(total)}", ha='center', va='bottom', fontsize=9)

    # 3. Add padding to the top of the y-axis so the labels aren't cut off
    axes[1, i].margins(y=0.3)

add_error_legend(fig, errors_plotted)

axes[0, 0].set_ylabel("Error count")
axes[1, 0].set_ylabel("Proportion of Total Errors")


# Adjust layout to prevent overlap, leaving room at the top for the legend
fig.tight_layout()
fig.subplots_adjust(bottom=0.15)
axes[1, 0].set_ylim(0, 110)
fig.savefig(OUTPUTS/'errors_per_model.png')

In [ ]:
N_SAMPLES = 3

bar_kwargs = dict(edgecolor='white', linewidth=0.5, kind='barh', stacked=True, legend=False)

fig, axes = plt.subplots(nrows=len(MODELS), ncols=2, figsize=(10, 10), sharex=True, sharey='row')

errors_plotted = []

for row_idx, model_name in enumerate(MODELS):
    model_df = df[df['model'] == model_name].copy()

    top_ids = set()
    for prompt_type in prompts:
        prompt_df = model_df[model_df['prompt'] == prompt_type]
        top_n_for_prompt = prompt_df.groupby('id').size().nlargest(N_SAMPLES).index.tolist()
        top_ids.update(top_n_for_prompt)

    # Convert to sorted list (Y-axis labels)
    top_ids = sorted(list(top_ids))

    # if a model had no errors, skip to avoid crashes
    if not top_ids:
        continue

    # Filter model data to just these IDs
    filtered_df = model_df[model_df['id'].isin(top_ids)]

    # Group the filtered data
    grouped = filtered_df.groupby(['prompt', 'id', 'error_type']).size().unstack(fill_value=0)

    for col_idx, prompt_type in enumerate(prompts):
        ax = axes[row_idx, col_idx]

        # Extract data or create empty frame if prompt had 0 errors for these IDs
        if prompt_type in grouped.index:
            plot_data = grouped.loc[prompt_type]
        else:
            plot_data = pd.DataFrame(columns=colours_code_labels)

        # reindex to ensure y-axis and stack order alignment
        plot_data = plot_data.reindex(index=top_ids, fill_value=0)
        plot_data = plot_data.reindex(columns=colours_code_labels, fill_value=0)

        # draw the bars
        plot_data.plot(ax=ax, color=list(ERROR_COLOURS.values()), **bar_kwargs)

        # register colours
        totals_per_error = plot_data.sum(axis=0)
        errors_plotted.extend(totals_per_error[totals_per_error > 0].index)

        # Add bar labels
        totals_per_id = plot_data.sum(axis=1)
        for i, total in enumerate(totals_per_id):
            ax.text(x=total, y=i, s=f' {total}', va='center', ha='left', fontsize=9)

        # Formatting for the grid
        ax.set_axisbelow(True)

        if row_idx == 0:
            ax.set_title(f"{prompt_type.capitalize()} code prompt")

    axes[row_idx, 0].set_ylabel(f"{model_name}\n\nTemplate ID")


for ax in axes[-1]:
    ax.set_xlabel("Error Count")

# Create the shared legend
add_error_legend(fig, errors_plotted)

# Adjust layout to prevent overlap, leaving room at the bottom for the legend
fig.tight_layout()
fig.subplots_adjust(bottom=0.1)

fig.savefig(OUTPUTS/'top_error_ids_per_model.png')


In [ ]:
def cleaunp_table(table: pd.DataFrame):
    keys = list(ERROR_COLOURS.keys())
    if 'Other' not in table.columns:
        keys = keys[:-1]
    table = table.reindex(columns=keys).rename(columns=ERROR_NAMES)

    return table

In [ ]:
for prompt in prompts:
    df_prompt = df[df.prompt == prompt]

    counts = df_prompt.groupby(['error_type', 'model']).size().reset_index(name='error_count')
    counts_table = counts.pivot(index='model', columns='error_type', values='error_count')
    counts_table = cleaunp_table(counts_table)
    counts_table = counts_table.fillna(0).astype(int)

    print(pandas_to_latex(counts_table, label=f"tab:{prompt}-code-error-types", caption=f"Error counts per error type for each model, {prompt} code prompt."))


In [ ]:
for prompt in prompts:
    df_prompt = df[df.prompt == prompt]

    counts = df_prompt.groupby(['error_type', 'model', 'id']).size().reset_index(name='error_count')

    top_indices = counts.groupby(['error_type', 'model'])['error_count'].idxmax()
    top_combinations = counts.loc[top_indices].copy()

    # create the formatted string: "TemplateID (Count)"
    top_combinations['cell_string'] = (
        top_combinations['id'].astype(str) +
        " (" +
        top_combinations['error_count'].astype(str) +
        ")"
    )

    # pivot the table
    top_id_table = top_combinations.pivot(
        index='model',
        columns='error_type',
        values='cell_string'
    )
    top_id_table = cleaunp_table(top_id_table)
    top_id_table = top_id_table.fillna('-')

    print(pandas_to_latex(top_id_table, label=f"tab:{prompt}-code-errors-types-top-id", caption=f"Most frequent template IDs per error type for each model, {prompt} code prompt. Total instances in parentheses."))

In [ ]:
def print_question(template_id, model, error_type, prompt_type, choice: int | None = None):
    qdf = df[(df.id == template_id) * (df.model == model) * (df.error_type == error_type) * (df.prompt == prompt_type)]
    if not len(qdf):
        print("Empty filter result")
        return

    row = qdf.iloc[choice if choice is not None else 0]

    sep = lambda s: f"{s}\n{len(s) * '-'}"

    print(sep("Question:"))
    print(row.question)

    print()
    print(sep("Answer:"))
    func_def, func_name = AnswerExtractor.extract_function_definition(row.full_response)
    print(func_def)

    print(sep("Issue:"))
    print(AnswerExtractor.try_running_function(func_def, func_name)[-1])

In [19]:
print_question(94, 'Phi-3-mini-128k-instruct', 'NAME_ERROR', 'short')

Question:
---------
1000 people apply for a job at Amazon. Of the people that apply, only 40% receive interviews. Of those who receive interviews, 49% receive a job offer. Of those who receive a job offer, two-fourths of the people accept the position. How many people accept the position?

Answer:
-------
def solution():

    applicants = 1000
    interview_rate = 0.4
    offer_rate = 0.49
    acceptance_rate = 0.5

    # first, calculate the number of people who receive interviews
    interviewees = applicants * interview_rate
    # next, calculate the number of people who receive job offers
    offer_recipients = interviewees * offer_rate
    # finally, calculate the number of people who accept the job offer
    acceptors = offer_recipients * acceptance_rate
    return int(acceptor)



Issue:
------
NameError when running extracted function: name 'acceptor' is not defined


In [20]:
print_question(80, 'Phi-3.5-mini-instruct', 'NOT_A_NUMBER', 'long')

Question:
---------
The vending machines sell cookies for 50 cents and pretzels for 75 cents. Jamal spent $1200 and got 6 bags of cookies and had 7% of his money left in change. How many pretzels did he buy?

Answer:
-------
def solution():

    # given:
    total_spent = 1200  # total amount of money Jamal spent
    cookies_bought = 6  # number of bags of cookies Jamal bought
    cookie_price = 0.50  # price of one bag of cookies
    change_percentage = 7  # percentage of money left in change

    # to find: number of pretzels Jamal bought

    # solution:
    # first, calculate the total cost of the cookies
    cookies_cost_total = cookies_bought * cookie_price
    # next, calculate the amount of money Jamal spent on cookies
    money_spent_on_cookies = cookies_cost_total
    # calculate the amount of money Jamal had left after buying cookies
    money_left_after_cookies = total_spent - money_spent_on_cookies
    # calculate the amount of money Jamal had left as a percentage of the t